# 03 — XGBoost and class-imbalance experiments (Member 3)

This notebook **reads** the outputs of `scripts/run_xgboost_experiments.py`; it trains nothing.
Regenerate the inputs from the repository root with:

```powershell
uv run python scripts/run_xgboost_experiments.py
```

| Experiment | Question | Models |
|---|---|---|
| **E3** | Is tuning worthwhile? | `xgboost_default` vs `xgboost_standard_tuned` |
| **E2** | Does imbalance handling help? | `standard` vs `weighted` (`scale_pos_weight`) vs `smote`, all tuned with the same budget |
| **E1** (XGBoost part) | How does XGBoost compare with the other models? | XGBoost vs Logistic Regression / Random Forest, when their predictions are available |

**Scope.** All numbers are on the shared **validation** partition (22,500 rows). Hyperparameters were
selected by stratified 3-fold CV on the **training** partition only. The test partition is never read.
Class 1 = serious delinquency (90+ days past due) within two years, not default or financial loss.

In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.calibration import calibration_curve
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay

# Support running from either the repository root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.evaluate import PREDICTION_COLUMNS, evaluate_probabilities

TABLES = ROOT / "results" / "tables" / "member3"
ARTIFACTS = ROOT / "artifacts" / "member3"
FIGURES = ROOT / "results" / "figures" / "member3"
FIGURES.mkdir(parents=True, exist_ok=True)

MODELS = {
    "xgboost_default": "XGB default (untuned)",
    "xgboost_standard_tuned": "XGB tuned - standard",
    "xgboost_weighted_tuned": "XGB tuned - scale_pos_weight",
    "xgboost_smote_tuned": "XGB tuned - SMOTE",
}
pd.set_option("display.precision", 4)

## 1. Run provenance

Refuse to analyse smoke-run outputs and record exactly which code and data produced the results.

In [ ]:
meta = yaml.safe_load((ARTIFACTS / "configuration.yaml").read_text(encoding="utf-8"))
assert not meta["run"]["smoke_run"], "Smoke-run outputs: rerun the full experiment before reporting"
assert meta["test_evaluated"] is False

pd.Series({
    "git_revision": meta["git_revision"],
    "uncommitted_changes": meta["git_uncommitted_changes"],
    "dataset_fingerprint": meta["dataset_fingerprint"][:16] + "...",
    "seed": meta["seed"],
    "cv_folds": meta["run"]["cv"],
    "candidates_per_strategy": meta["run"]["n_iter"],
    "scale_pos_weight (train)": round(meta["scale_pos_weight"], 4),
    **{f"version {k}": v for k, v in meta["library_versions"].items()},
}, name="value").to_frame()

## 2. Results overview (validation)

`cv_average_precision` is the mean over training folds for the selected configuration.
Precision / recall / F1 use the fixed reporting threshold 0.5; threshold choice belongs to the
cost analysis (Member 5).

In [ ]:
summary = pd.read_csv(TABLES / "summary.csv").set_index("model_name")
metrics = json.loads((TABLES / "metrics.json").read_text(encoding="utf-8"))
columns = ["experiment", "cv_average_precision", "roc_auc", "average_precision",
           "brier_score", "precision", "recall", "f1", "fit_seconds"]
summary.loc[list(MODELS), columns]

## 3. E3 — Is tuning worthwhile?

Same strategy (`standard`), library defaults vs randomized search.

In [ ]:
e3 = summary.loc[["xgboost_default", "xgboost_standard_tuned"],
                 ["roc_auc", "average_precision", "brier_score"]]
e3.loc["gain from tuning"] = e3.iloc[1] - e3.iloc[0]
display(e3)
pd.Series(metrics["xgboost_standard_tuned"]["best_params"], name="selected hyperparameters",
          dtype=object).to_frame()

### Sensitivity to hyperparameters

Each point is one sampled configuration, scored by mean CV average precision on the training folds.
The spread shows how much a poor configuration would cost.

In [ ]:
cv = {s: pd.read_csv(TABLES / f"cv_results_{s}.csv") for s in ["standard", "weighted", "smote"]}

fig, ax = plt.subplots(figsize=(8, 4.5))
for i, (strategy, frame) in enumerate(cv.items()):
    scores = frame["mean_test_score"]
    jitter = np.random.default_rng(0).uniform(-0.12, 0.12, len(scores))
    ax.scatter(np.full(len(scores), i) + jitter, scores, alpha=0.7)
    ax.hlines(scores.max(), i - 0.25, i + 0.25, color="black")
ax.set_xticks(range(len(cv)), list(cv))
ax.set_ylabel("Mean CV average precision (training folds)")
ax.set_title("Randomized search: every sampled configuration (bar = selected)")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES / "cv_sensitivity.png", dpi=150)
plt.show()

pd.DataFrame({s: f["mean_test_score"].describe()[["min", "50%", "max"]] for s, f in cv.items()}).T

In [ ]:
param_cols = [c for c in cv["standard"].columns if c.startswith("param_")]
cv["standard"][["rank_test_score", "mean_test_score", "std_test_score", *param_cols]].head(5)

## 4. E2 — Does imbalance handling help?

All three strategies were tuned with the same seed, so the search evaluated the **same 20 candidate
configurations** for each; only the imbalance handling differs. Ranking quality (ROC-AUC, AP) and
probability quality (Brier, mean predicted probability vs observed rate) are reported separately.

In [ ]:
predictions = {name: pd.read_csv(TABLES / f"{name}_validation_predictions.csv") for name in MODELS}

# Contract and alignment checks before any comparison.
splits = json.loads((ROOT / "data" / "processed" / "splits.json").read_text(encoding="utf-8"))["splits"]
reference = predictions["xgboost_default"]
for name, frame in predictions.items():
    assert list(frame.columns) == PREDICTION_COLUMNS, name
    assert frame["sample_id"].tolist() == splits["validation"], name
    assert frame["split"].eq("validation").all(), name
    np.testing.assert_array_equal(frame["y_true"], reference["y_true"])
observed_rate = reference["y_true"].mean()
print(f"Validation rows: {len(reference)}, observed delinquency rate: {observed_rate:.4f}")

e2 = summary.loc[["xgboost_standard_tuned", "xgboost_weighted_tuned", "xgboost_smote_tuned"],
                 ["roc_auc", "average_precision", "brier_score", "recall", "fit_seconds"]]
e2["mean_predicted_probability"] = [predictions[n]["y_pred_proba"].mean() for n in e2.index]
e2["observed_rate"] = observed_rate
e2

## 5. ROC and precision-recall curves

On imbalanced data the precision-recall curve is the more informative view (Saito & Rehmsmeier, 2015);
the dashed line is the no-skill level (observed positive rate).

In [ ]:
fig, (ax_roc, ax_pr) = plt.subplots(1, 2, figsize=(13, 5.5))
for name, label in MODELS.items():
    frame = predictions[name]
    RocCurveDisplay.from_predictions(frame["y_true"], frame["y_pred_proba"], name=label, ax=ax_roc)
    PrecisionRecallDisplay.from_predictions(frame["y_true"], frame["y_pred_proba"], name=label, ax=ax_pr)
ax_roc.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax_roc.set_title("ROC - validation")
ax_pr.axhline(observed_rate, color="gray", linestyle="--", label=f"No skill = {observed_rate:.3f}")
ax_pr.set_title("Precision-recall - validation")
for ax in (ax_roc, ax_pr):
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES / "roc_pr_curves.png", dpi=150)
plt.show()

## 6. Probability reliability (preliminary)

A quick reliability diagram to document **why** imbalance handling changes probability meaning.
Formal calibration (E4) is owned by Member 4; nothing is calibrated here.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
for name in ["xgboost_standard_tuned", "xgboost_weighted_tuned", "xgboost_smote_tuned"]:
    frame = predictions[name]
    observed, predicted = calibration_curve(frame["y_true"], frame["y_pred_proba"],
                                            n_bins=10, strategy="quantile")
    ax.plot(predicted, observed, marker="o", label=MODELS[name])
ax.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Perfect reliability")
ax.set_xlabel("Mean predicted probability (decile bins)")
ax.set_ylabel("Observed delinquency rate")
ax.set_title("Reliability diagram - validation")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(FIGURES / "reliability.png", dpi=150)
plt.show()

## 7. Confusion matrices at the reporting threshold 0.5

Not comparable across strategies: `scale_pos_weight` shifts every probability upward, so the same
threshold means a different decision rule.

In [ ]:
fig, axes = plt.subplots(1, len(MODELS), figsize=(18, 4.2))
for ax, (name, label) in zip(axes, MODELS.items()):
    frame = predictions[name]
    ConfusionMatrixDisplay.from_predictions(
        frame["y_true"], (frame["y_pred_proba"] >= 0.5).astype(int), labels=[0, 1],
        display_labels=["No delinquency", "Delinquency"], cmap="Blues", colorbar=False, ax=ax)
    ax.set_title(label, fontsize=10)
fig.suptitle("Confusion matrices - threshold 0.5 (rows = true, columns = predicted)")
plt.tight_layout()
fig.savefig(FIGURES / "confusion_matrices.png", dpi=150)
plt.show()

## 8. Feature importance of the selected model (gain)

Total split gain inside the trees of `xgboost_standard_tuned`, including the two missing-value
indicators added by the shared preprocessing. Gain is model-internal and **not causal**; local and
global SHAP explanations are Member 4's deliverable.

In [ ]:
# Trusted artifact produced by this project's own script.
pipeline = joblib.load(ARTIFACTS / "xgboost_standard_tuned.joblib")
feature_names = pipeline.named_steps["preprocessing"].get_feature_names_out()
gain = pipeline.named_steps["model"].get_booster().get_score(importance_type="total_gain")
importance = (pd.Series({feature_names[int(key[1:])]: value for key, value in gain.items()})
              .reindex(feature_names, fill_value=0.0))
importance = (importance / importance.sum()).sort_values()

fig, ax = plt.subplots(figsize=(8, 5))
importance.plot.barh(ax=ax)
ax.set_xlabel("Share of total gain")
ax.set_title("XGBoost tuned (standard): gain importance")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
fig.savefig(FIGURES / "feature_importance_gain.png", dpi=150)
plt.show()

## 9. E1 — Comparison with the other models

Loads other members' validation predictions **if they exist locally** (generated outputs are not in
Git). Every model uses the same validation rows, checked through `sample_id`.

In [ ]:
candidates = {
    "Logistic Regression - baseline (M1)": ROOT / "results/tables/baselines/logistic_regression_validation_predictions.csv",
    "Logistic Regression - tuned (M2)": ROOT / "results/tables/member2/logistic_tuned_validation_predictions.csv",
    "Random Forest - tuned (M2)": ROOT / "results/tables/member2/random_forest_tuned_validation_predictions.csv",
    "XGBoost - tuned standard (M3)": TABLES / "xgboost_standard_tuned_validation_predictions.csv",
}
rows, missing = {}, []
for label, path in candidates.items():
    if not path.exists():
        missing.append(label)
        continue
    frame = pd.read_csv(path)
    assert frame["sample_id"].tolist() == splits["validation"], label
    scores = evaluate_probabilities(frame["y_true"], frame["y_pred_proba"])["probability_metrics"]
    rows[label] = {"roc_auc": scores["roc_auc"], "average_precision": scores["pr_auc_average_precision"],
                   "brier_score": scores["brier_score"]}
if missing:
    print("Not available locally (ask the owner or run their script):", ", ".join(missing))
pd.DataFrame(rows).T.sort_values("average_precision", ascending=False)

## 10. Findings (validation only, single seed)

Figures from the run recorded in Section 1 (commit `912ce05`, 20 candidates x 3 folds per strategy).

- **E3 — tuning helps moderately.** Validation AP rises from 0.378 to 0.404 and ROC-AUC from 0.861 to
  0.870. CV scores across sampled configurations range from about 0.34 to 0.40, so a poor configuration
  costs more than tuning gains over the defaults.
- **E2 — imbalance handling does not improve ranking.** `scale_pos_weight` gives the same AP / ROC-AUC
  as standard training (same configuration selected) but shifts every probability upward (mean
  predicted probability about 0.32 vs an observed rate of 0.067), tripling the Brier score. Its high
  recall at 0.5 is a threshold effect, reproducible by lowering the threshold of the standard model.
  SMOTE is lower on AP (0.389), about 2.7x slower to tune, and mildly overestimates risk (mean
  predicted probability 0.086). The standard model's reliability curve stays close to the diagonal.
- **Drivers (gain, not causal).** Past-delinquency counts and revolving utilization dominate the
  splits; the delinquency counts include the 96/98 values flagged in the EDA, which trees can isolate.
  This should be cross-checked with SHAP (Member 4).
- **E1 — XGBoost and Random Forest are effectively tied.** Member 2 reported RF validation AP 0.401 /
  ROC-AUC 0.869; the XGBoost gap (+0.003 AP) is below the CV fold standard deviation (about 0.004), so no
  superiority claim is made. Both clearly outperform Logistic Regression.

**Handoff.** Member 4: `xgboost_standard_tuned` is the recommended XGBoost candidate (best AP and Brier,
mean probability matching the observed rate); `xgboost_weighted_tuned` illustrates distorted
probabilities for E4. Member 5: imbalance is better handled at decision time through the threshold.

**Limitations.** One seed and one validation partition; selecting the best of several models on the
same validation set makes its score slightly optimistic; threshold metrics at 0.5 are not comparable
across strategies; gain importance is not causal. Final claims require the locked test evaluation.